# 08 - Cross-Project Evaluation

Đây là thí nghiệm quan trọng nhất cho chữ **cross-project**.

Mỗi vòng:
1. giữ nguyên 1 project làm test;
2. train bằng các project còn lại;
3. đánh giá Cox và RSF trên project chưa từng xuất hiện trong train.

`FAST_MODE=True` chỉ chạy vài fold để kiểm tra code.  
**Trước khi lấy số liệu báo cáo, phải đặt `FAST_MODE=False`.**

In [1]:
from pathlib import Path
import sys, yaml, pandas as pd, numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

with open(ROOT / "configs" / "experiment.yaml", "r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

print("ROOT =", ROOT)
print("random_seed =", CFG["random_seed"])

ROOT = d:\VNUK\Eureka 2026\Eureka_2026
random_seed = 42


In [2]:
from src.features import feature_spec, sample_training_rows
from src.models import fit_cox, fit_rsf
from src.evaluation import evaluate_survival_model

MODE = "day0"
FAST_MODE = True   # <-- đổi FALSE khi chạy kết quả cuối

data = pd.read_parquet(ROOT / "data" / "processed" / f"model_{MODE}.parquet")
projects = sorted(data["project"].astype(str).unique())

if FAST_MODE:
    heldout_projects = projects[: min(3, len(projects))]
    print("SMOKE TEST ONLY:", heldout_projects)
else:
    heldout_projects = projects
    print("FINAL LOPO folds:", len(heldout_projects))

SMOKE TEST ONLY: ['CONFSERVER', 'FLEX', 'HIVE']


In [4]:
strict = bool(CFG["features"]["strict_no_leakage"])
landmark = int(CFG["features"]["landmark_days"])
numeric_cols, categorical_cols = feature_spec(
    MODE,
    strict_no_leakage=strict,
    landmark_days=landmark,
)

max_rows = 20000
horizons = CFG["evaluation"]["horizons_days"]

rows = []

for fold, heldout in enumerate(heldout_projects, start=1):
    print(f"\n[{fold}/{len(heldout_projects)}] Hold out:", heldout)

    train = data[data["project"].astype(str) != heldout].copy()
    test = data[data["project"].astype(str) == heldout].copy()
    test = test.sample(n=min(5000, len(test)), random_state=42)

    train_fit = sample_training_rows(
        train,
        max_rows=max_rows,
        random_state=CFG["random_seed"] + fold,
    )

    # Cox
    cox = fit_cox(
        train_fit,
        numeric_cols,
        categorical_cols,
        alpha=float(CFG["models"]["cox"]["alpha"]),
    )
    m_cox = evaluate_survival_model(cox, train_fit, test, horizons)
    rows.append({
        "heldout_project": heldout,
        "model": "CoxPH",
        "train_rows": len(train_fit),
        "test_rows": len(test),
        **m_cox,
    })

    # RSF
    rsf_cfg = CFG["models"]["rsf"].copy()
    rsf_cfg["random_state"] = CFG["random_seed"] + fold
    rsf_cfg["n_estimators"] = 20
    rsf_cfg["max_depth"] = 10
    rsf_cfg["n_jobs"] = 1
    rsf = fit_rsf(
        train_fit,
        numeric_cols,
        categorical_cols,
        **rsf_cfg,
    )
    m_rsf = evaluate_survival_model(rsf, train_fit, test, horizons)
    rows.append({
        "heldout_project": heldout,
        "model": "RSF",
        "train_rows": len(train_fit),
        "test_rows": len(test),
        **m_rsf,
    })

cross = pd.DataFrame(rows)
cross


[1/3] Hold out: CONFSERVER


d:\VNUK\Eureka 2026\Eureka_2026\.venv\Lib\site-packages\sksurv\linear_model\coxph.py:201: RuntimeWarning: overflow encountered in exp
  risk_set += np.exp(xw[k])
d:\VNUK\Eureka 2026\Eureka_2026\.venv\Lib\site-packages\sksurv\linear_model\coxph.py:198: RuntimeWarning: overflow encountered in exp
  risk_set2 += np.exp(xw[k])



[2/3] Hold out: FLEX


d:\VNUK\Eureka 2026\Eureka_2026\.venv\Lib\site-packages\sksurv\linear_model\coxph.py:201: RuntimeWarning: overflow encountered in exp
  risk_set += np.exp(xw[k])
d:\VNUK\Eureka 2026\Eureka_2026\.venv\Lib\site-packages\sksurv\linear_model\coxph.py:198: RuntimeWarning: overflow encountered in exp
  risk_set2 += np.exp(xw[k])



[3/3] Hold out: HIVE


,heldout_project,model,train_rows,test_rows,n_test,events_test,harrell_c,ipcw_c,auc_30,auc_60,auc_90,mean_dynamic_auc,ibs
0,CONFSERVER,CoxPH,20000,5000,5000,4218,0.589194,0.582418,0.632169,0.633290,0.631689,0.632268,0.227410
1,CONFSERVER,RSF,20000,5000,5000,4218,0.604803,0.593980,0.670400,0.674900,0.676139,0.671399,0.222568
2,FLEX,CoxPH,20000,5000,5000,4431,0.512893,0.510908,0.536380,0.529816,0.527513,0.534813,0.189499
3,FLEX,RSF,20000,5000,5000,4431,0.515821,0.513379,0.540985,0.533028,0.528191,0.538928,0.183659
4,HIVE,CoxPH,20000,5000,5000,3587,0.600912,0.599464,0.647059,0.651216,0.651045,0.647714,0.184524
5,HIVE,RSF,20000,5000,5000,3587,0.602919,0.601481,0.649441,0.654036,0.653069,0.650126,0.189426


In [5]:
table_dir = ROOT / "results" / "tables"
table_dir.mkdir(parents=True, exist_ok=True)

suffix = "SMOKE" if FAST_MODE else "FINAL"
out_file = table_dir / f"cross_project_{MODE}_{suffix}.csv"
cross.to_csv(out_file, index=False)

metric_cols = [
    c for c in ["harrell_c", "ipcw_c", "auc_30", "auc_60", "auc_90", "ibs"]
    if c in cross.columns
]

summary = (
    cross.groupby("model")[metric_cols]
    .agg(["mean", "std"])
)

print("Saved:", out_file)
summary

Saved: d:\VNUK\Eureka 2026\Eureka_2026\results\tables\cross_project_day0_SMOKE.csv


harrell_c              ipcw_c              auc_30              auc_60  \
           mean       std      mean       std      mean       std      mean   
model                                                                         
CoxPH  0.567666  0.047795  0.564263  0.046987  0.605203  0.060065  0.604774   
RSF    0.574515  0.050838  0.569613  0.048845  0.620275  0.069463  0.620655   

                   auc_90                 ibs            
            std      mean       std      mean       std  
model                                                    
CoxPH  0.065531  0.603416  0.066442  0.200477  0.023456  
RSF    0.076600  0.619133  0.079598  0.198551  0.020998

### Cách đọc

- `Harrell C / IPCW C / AUC`: cao hơn tốt hơn.
- `IBS`: thấp hơn tốt hơn.
- Không chỉ báo mean; nên báo **mean ± SD giữa các held-out project** và giữ bảng từng project.